# 03 · The finding: health publishers vs the national corridor

**Reconciliation note, which I keep in force all the way through.** I cannot rebuild the National Data
Library's national figure. Their scope was the whole catalogue, tested in June 2026, before the
cleanup. Mine is six health publishers, tested on the date of my snapshot, *after* the cleanup.
So I quote their figures in one table and compute mine in another, and I never blend the two
into a single number. The national figures below are as quoted in "The National Data Library –
cleaning up data.gov.uk" (12 Nov 2026). I have not been able to retrieve that post myself.

**Classes, mirroring the NDL's split:**
- `ok`: 2xx, and not a bot challenge
- `hard_dead`: 404, 410, DNS or connection failure
- `uncertain`: 403, timeout, 400, 500, 429, any other status, and unresolved WAF challenges

A browser re-check moves a link out of `uncertain` only if the browser actually reached the page.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from hclr import config


In [2]:
import json, time, requests
from hclr import classify as C
resources = pd.read_parquet(config.SILVER / "resources.parquet")
datasets = pd.read_parquet(config.SILVER / "datasets.parquet")
links = C.classify(pd.read_parquet(config.SILVER / "link_checks.parquet"))
labels = {k: v["label"] for k, v in config.PUBLISHERS.items()}
print(len(resources), "resource links ->", len(links), "distinct URLs classified")

4305 resource links -> 3111 distinct URLs classified


## Corroborating the DNS failures

Through this machine's egress proxy, a DNS or connect failure shows up as a `502` on the CONNECT
tunnel. I check each such host once against public DNS-over-HTTPS, so I can say which are truly
unregistered (NXDOMAIN) and which still resolve but refuse connections. Both count as `hard_dead`
under the NDL's rules. The split just tells me which kind of death it is.

In [3]:
dns_out = config.BRONZE / "linkcheck" / resources["snapshot_utc"].iloc[0] / "dns_corroboration.json"
fail_hosts = sorted(links.loc[links["http_reason"] == "dns_or_connection_failure", "host"].unique())
if dns_out.exists():
    dns = json.loads(dns_out.read_text())
else:
    dns = {}
    for h in fail_hosts:
        try:
            r = requests.get("https://dns.google/resolve", params={"name": h.split(":")[0], "type": "A"},
                             headers={"User-Agent": config.USER_AGENT}, timeout=15)
            dns[h] = {"Status": r.json().get("Status"), "Answer": r.json().get("Answer")}
        except Exception as e:
            dns[h] = {"Status": None, "error": str(e)}
        time.sleep(1)
    dns_out.write_text(json.dumps(dns, indent=2))
links["dns_rcode"] = links["host"].map(lambda h: (dns.get(h) or {}).get("Status"))
pd.Series({h: ("NXDOMAIN" if v.get("Status") == 3 else "no address record (NOERROR, empty answer)" if v.get("Status") == 0 and not v.get("Answer")
               else "resolves" if v.get("Status") == 0 else v.get("Status"))
           for h, v in dns.items()}).value_counts()

no address record (NOERROR, empty answer)    1
Name: count, dtype: int64

## Reading Cloudflare's 409s

A `409` from Cloudflare usually means Cloudflare is fronting a hostname whose own origin
DNS has gone (Cloudflare error 1001, "DNS resolution error"). By the NDL's rules that's a DNS
failure, and so `hard_dead`, even though the status code hides it. For each host that returned a
Cloudflare 409, I fetch **one** of its error pages and record the Cloudflare error code. I keep
the raw page in Bronze. If it's 1001, every 409 on that host counts as a DNS failure. The error
belongs to the hostname, not to the path.

In [4]:
import re
cf_out = config.BRONZE / "linkcheck" / resources["snapshot_utc"].iloc[0] / "cloudflare_409_corroboration.json"
cf409 = links[(links["http_status"] == 409) & links["head.server"].fillna("").str.contains("cloudflare", case=False)]
if cf_out.exists():
    cf = json.loads(cf_out.read_text())
else:
    cf = {}
    for host, g in cf409.groupby("host"):
        r = requests.get(g["url"].iloc[0], headers={"User-Agent": config.USER_AGENT}, timeout=(10, 30))
        title = re.search(r"<title>(.*?)</title>", r.text, re.S)
        code = re.search(r"errorCode:\s*(\d{4})", r.text)
        cf[host] = {"sample_url": g["url"].iloc[0], "status": r.status_code,
                    "title": title.group(1).strip() if title else None, "cloudflare_error": code.group(1) if code else None,
                    "fetched_utc": pd.Timestamp.now(tz="UTC").isoformat()}
        (cf_out.parent / f"cloudflare_409_{host}.html").write_text(r.text)
        time.sleep(2)
    cf_out.write_text(json.dumps(cf, indent=2))
dns_1001 = {h for h, v in cf.items() if v.get("cloudflare_error") == "1001"}
m = (links["http_status"] == 409) & links["host"].isin(dns_1001)
links.loc[m, ["http_class", "final_class"]] = "hard_dead"
links.loc[m, ["http_reason", "final_reason"]] = "cloudflare_1001_dns_resolution_error"
print({h: (v["cloudflare_error"], v["title"]) for h, v in cf.items()}, "->", int(m.sum()), "URLs reclassified")

{'www.ic.nhs.uk': ('1001', 'DNS resolution error | www.ic.nhs.uk | Cloudflare')} -> 102 URLs reclassified


## Joining back to every catalogue link

The NDL counted *links*, so my headline unit is also the resource link. Each URL's
result applies to every catalogue entry that points at it. I show a distinct-URL view
next to it, because some publishers repeat the same URL across many resources.

In [5]:
res = resources.merge(links, on="url", how="left", suffixes=("", "_chk"))
assert res["final_class"].notna().all(), "every resource link must have a result"
res = res.merge(datasets[["dataset_id", "harvest_marker", "metadata_modified"]], on="dataset_id", how="left")
res["publisher"] = res["publisher_slug"].map(labels)
res.to_parquet(config.SILVER / "resource_link_status.parquet", index=False)

final = C.summarise(res, "publisher")
auto = C.summarise(res, "publisher", cls="http_class")
distinct = C.summarise(res.drop_duplicates(["publisher", "url"]), "publisher")
config.GOLD.mkdir(parents=True, exist_ok=True)
for name, t in {"by_publisher_final": final, "by_publisher_http_only": auto, "by_publisher_distinct_urls": distinct}.items():
    t.to_parquet(config.GOLD / f"{name}.parquet"); t.round(1).to_csv(config.GOLD / f"{name}.csv")
show = ["links", "ok", "hard_dead", "uncertain", "ok_pct", "hard_dead_pct", "uncertain_pct",
        "hard_dead_ci_lo", "hard_dead_ci_hi", "broken_floor_pct", "broken_ceiling_pct"]
print("HEALTH PUBLISHERS - resource links, after browser re-check (my figures)")
final[show].round(1)

HEALTH PUBLISHERS - resource links, after browser re-check (my figures)


final_class,links,ok,hard_dead,uncertain,ok_pct,hard_dead_pct,uncertain_pct,hard_dead_ci_lo,hard_dead_ci_hi,broken_floor_pct,broken_ceiling_pct
publisher,,,,,,,,,,,
"NHS Digital (merged into NHS England, 2023)",2489,660,96,1733,26.5,3.9,69.6,3.2,4.7,3.9,73.5
Department of Health and Social Care,1160,388,10,762,33.4,0.9,65.7,0.5,1.6,0.9,66.6
NHS Business Services Authority,198,198,0,0,100.0,0.0,0.0,0.0,1.9,0.0,0.0
NHS England,439,419,0,20,95.4,0.0,4.6,0.0,0.9,0.0,4.6
Public Health England (closed 2021; to UKHSA/OHID),17,17,0,0,100.0,0.0,0.0,0.0,18.4,0.0,0.0
UK Health Security Agency,2,2,0,0,100.0,0.0,0.0,0.0,65.8,0.0,0.0


In [6]:
print("HTTP pass only, before any browser re-check (the analogue of the NDL's 'possible errors' stage)")
auto[show].round(1)

HTTP pass only, before any browser re-check (the analogue of the NDL's 'possible errors' stage)


http_class,links,ok,hard_dead,uncertain,ok_pct,hard_dead_pct,uncertain_pct,hard_dead_ci_lo,hard_dead_ci_hi,broken_floor_pct,broken_ceiling_pct
publisher,,,,,,,,,,,
"NHS Digital (merged into NHS England, 2023)",2489,659,96,1734,26.5,3.9,69.7,3.2,4.7,3.9,73.5
Department of Health and Social Care,1160,386,10,764,33.3,0.9,65.9,0.5,1.6,0.9,66.7
NHS Business Services Authority,198,198,0,0,100.0,0.0,0.0,0.0,1.9,0.0,0.0
NHS England,439,12,0,427,2.7,0.0,97.3,0.0,0.9,0.0,97.3
Public Health England (closed 2021; to UKHSA/OHID),17,16,0,1,94.1,0.0,5.9,0.0,18.4,0.0,5.9
UK Health Security Agency,2,2,0,0,100.0,0.0,0.0,0.0,65.8,0.0,0.0


In [7]:
print("Distinct URLs per publisher (each URL counted once)")
distinct[show].round(1)

Distinct URLs per publisher (each URL counted once)


final_class,links,ok,hard_dead,uncertain,ok_pct,hard_dead_pct,uncertain_pct,hard_dead_ci_lo,hard_dead_ci_hi,broken_floor_pct,broken_ceiling_pct
publisher,,,,,,,,,,,
"NHS Digital (merged into NHS England, 2023)",2166,627,95,1444,28.9,4.4,66.7,3.6,5.3,4.4,71.1
Department of Health and Social Care,670,362,8,300,54.0,1.2,44.8,0.6,2.3,1.2,46.0
NHS Business Services Authority,193,193,0,0,100.0,0.0,0.0,0.0,2.0,0.0,0.0
NHS England,74,72,0,2,97.3,0.0,2.7,0.0,4.9,0.0,2.7
Public Health England (closed 2021; to UKHSA/OHID),9,9,0,0,100.0,0.0,0.0,0.0,29.9,0.0,0.0
UK Health Security Agency,2,2,0,0,100.0,0.0,0.0,0.0,65.8,0.0,0.0


## The national backdrop, quoted and kept separate

In [8]:
ndl = pd.DataFrame([
    ("links tested (June 2026)", 250_260, None),
    ("datasets", 51_203, None),
    ("possible errors", 89_217, 100 * 89_217 / 250_260),
    ("  confident dead: 404/410/DNS/connection", 45_681, 100 * 45_681 / 250_260),
    ("  needing investigation: 403/timeout/400/500/429", 43_536, 100 * 43_536 / 250_260),
    ("links removed", 60_818, 100 * 60_818 / 250_260),
    ("broken rate before (as stated)", None, 30.0),
    ("broken rate after (as stated)", None, 7.0),
], columns=["NDL national figure (quoted)", "count", "pct_of_links_tested"])
ndl.round(1)

,NDL national figure (quoted),count,pct_of_links_tested
0,links tested (June 2026),250260.0,NaN
1,datasets,51203.0,NaN
2,possible errors,89217.0,35.6
3,confident dead: 404/410/DNS/connection,45681.0,18.3
4,needing investigation: 403/timeout/400/500/429,43536.0,17.4
5,links removed,60818.0,24.3
6,broken rate before (as stated),NaN,30.0
7,broken rate after (as stated),NaN,7.0


## Why links fail, per publisher

In [9]:
(res[res["final_class"] != "ok"].groupby(["publisher", "final_class", "final_reason"]).size()
   .rename("links").reset_index().sort_values(["publisher", "links"], ascending=[True, False]))

,publisher,final_class,final_reason,links
3,Department of Health and Social Care,uncertain,waf_challenge_browser_also_blocked,757
0,Department of Health and Social Care,hard_dead,cloudflare_1001_dns_resolution_error,7
2,Department of Health and Social Care,uncertain,not_an_http_url,5
1,Department of Health and Social Care,hard_dead,dns_or_connection_failure,3
7,"NHS Digital (merged into NHS England, 2023)",uncertain,waf_challenge_browser_also_blocked,1722
4,"NHS Digital (merged into NHS England, 2023)",hard_dead,cloudflare_1001_dns_resolution_error,96
5,"NHS Digital (merged into NHS England, 2023)",uncertain,browser_403,8
6,"NHS Digital (merged into NHS England, 2023)",uncertain,not_an_http_url,3
8,NHS England,uncertain,waf_challenge_browser_also_blocked,20


## Diagnostics that change what "ok" and "uncertain" mean

These don't change any class. They tell me how much weight each class can bear.

In [10]:
diag = res.groupby("publisher").agg(
    links=("url", "size"),
    ok=("final_class", lambda s: int((s == "ok").sum())),
    ok_but_redirects_to_homepage=("diag_redirect_to_homepage", "sum"),
    ok_on_a_different_domain=("diag_domain_changed", lambda s: int((s & (res.loc[s.index, "final_class"] == "ok")).sum())),
    web_archive_links=("diag_web_archive_link", "sum"),
    legacy_domain_links=("diag_legacy_domain", "sum"),
    browser_resolved=("resolved_by", lambda s: int((s == "browser").sum())),
    from_harvested_datasets=("harvest_marker", "sum"),
)
diag.to_csv(config.GOLD / "diagnostics_by_publisher.csv"); diag

,links,ok,ok_but_redirects_to_homepage,ok_on_a_different_domain,web_archive_links,legacy_domain_links,browser_resolved,from_harvested_datasets
publisher,,,,,,,,
Department of Health and Social Care,1160,388,10,1,757,7,2,0
NHS Business Services Authority,198,198,0,0,0,0,0,0
"NHS Digital (merged into NHS England, 2023)",2489,660,0,6,43,1428,9,250
NHS England,439,419,0,3,0,0,407,0
Public Health England (closed 2021; to UKHSA/OHID),17,17,0,0,0,0,1,0
UK Health Security Agency,2,2,0,0,0,0,0,0


## Dataset-level view

Readers think in datasets, not links. How many listings have at least one hard-dead link,
and how many have nothing left to follow at all?

In [11]:
ds_status = res.groupby(["publisher", "dataset_id"])["final_class"].agg(
    any_dead=lambda s: (s == "hard_dead").any(), all_ok=lambda s: (s == "ok").all()).reset_index()
ds_tab = ds_status.groupby("publisher").agg(datasets_with_links=("dataset_id", "size"),
                                            with_any_hard_dead=("any_dead", "sum"),
                                            all_links_ok=("all_ok", "sum"))
zero = datasets.assign(publisher=datasets["publisher_slug"].map(labels)).groupby("publisher")["num_resources"].agg(
    total_listings="size", listings_with_no_links=lambda s: int((s == 0).sum()))
ds_tab = zero.join(ds_tab).fillna(0).astype(int)
ds_tab.to_csv(config.GOLD / "datasets_by_publisher.csv"); ds_tab

,total_listings,listings_with_no_links,datasets_with_links,with_any_hard_dead,all_links_ok
publisher,,,,,
Department of Health and Social Care,164,35,129,9,23
NHS Business Services Authority,12,0,12,0,12
"NHS Digital (merged into NHS England, 2023)",475,78,397,19,29
NHS England,37,0,37,0,34
Public Health England (closed 2021; to UKHSA/OHID),134,128,6,0,6
UK Health Security Agency,1,0,1,0,1


## Empty listings: the cleanup's footprint?

A listing with no links left survives as a title and metadata only. If its `metadata_modified`
falls in July 2026, that's consistent with the cleanup having removed its links. I can't *prove*
that, because CKAN's activity stream (which would show the edit) returns 403 to the public API.

In [12]:
datasets["touched_jul_2026"] = datasets["metadata_modified"].dt.strftime("%Y-%m").eq("2026-07")
shells = pd.crosstab(datasets["publisher_slug"].map(labels),
                     [datasets["num_resources"].eq(0).map({True: "no links", False: "has links"}),
                      datasets["touched_jul_2026"].map({True: "modified Jul 2026", False: "modified earlier/later"})])
shells.to_csv(config.GOLD / "empty_listings_by_publisher.csv"); shells

num_resources                                              has links                                 no links                       
touched_jul_2026                                   modified Jul 2026 modified earlier/later modified Jul 2026 modified earlier/later
publisher_slug                                                                                                                      
Department of Health and Social Care                              18                    111                34                      1
NHS Business Services Authority                                    0                     12                 0                      0
NHS Digital (merged into NHS England, 2023)                       67                    330                12                     66
NHS England                                                        1                     36                 0                      0
Public Health England (closed 2021; to UKHSA/OHID)                 0                      6                 5                    123
UK Health Security Agency                                          0                      1                 0                      0

## Where the uncertainty actually sits

If the uncertain mass is spread across many hosts, it's link rot of unknown kind. If it piles up on a
few hosts that block automated checks, then it's a verifiability problem, which is a different story.

In [13]:
unc = res[res["final_class"] == "uncertain"]
by_host = unc.groupby(["publisher", "host"]).size().rename("uncertain_links").reset_index().sort_values("uncertain_links", ascending=False)
by_host["share_of_all_uncertain_pct"] = 100 * by_host["uncertain_links"] / len(unc)
by_host.to_csv(config.GOLD / "uncertain_by_host.csv", index=False); by_host.head(10).round(1)

,publisher,host,uncertain_links,share_of_all_uncertain_pct
10,"NHS Digital (merged into NHS England, 2023)",www.hscic.gov.uk,1332,53.0
2,Department of Health and Social Care,webarchive.nationalarchives.gov.uk,757,30.1
4,"NHS Digital (merged into NHS England, 2023)",digital.nhs.uk,288,11.5
12,"NHS Digital (merged into NHS England, 2023)",www.nchod.nhs.uk,58,2.3
8,"NHS Digital (merged into NHS England, 2023)",webarchive.nationalarchives.gov.uk,43,1.7
14,NHS England,www.england.nhs.uk,20,0.8
13,"NHS Digital (merged into NHS England, 2023)",www.nice.org.uk,5,0.2
0,Department of Health and Social Care,,3,0.1
1,Department of Health and Social Care,assets.publishing.service.gov.uk,2,0.1
3,"NHS Digital (merged into NHS England, 2023)",,2,0.1


## A sample for a human to check by hand

No automated method I'm willing to use gets past these walls. The only honest way to narrow
the uncertain band is a person with a normal browser checking a random sample. I draw it with a fixed
seed, stratified by publisher, so the sample is reproducible and can't be cherry-picked.

In [14]:
# Shuffle once with a fixed seed, then take the first 30 per publisher.
sample = unc.drop_duplicates("url").sample(frac=1, random_state=20261121).groupby("publisher_slug").head(30)
sample = sample[["publisher_slug", "dataset_name", "url", "host", "final_reason"]].assign(
    human_result="", human_notes="")
sample.to_csv(config.OUTPUTS / "manual_check_sample.csv", index=False)
sample.groupby("publisher_slug").size()

publisher_slug
department-of-health    30
nhs-digital             30
nhs-england              2
dtype: int64

## Harvested vs hand-published (NHS Digital is the only publisher here with harvest markers)

Caution: every harvested link here points at `digital.nhs.uk`, so this comparison confounds harvesting with host.

In [15]:
nd = res[res["publisher_slug"] == "nhs-digital"]
C.summarise(nd.assign(source=nd["harvest_marker"].map({True: "harvested", False: "not harvested"})), "source")[show].round(1)

final_class,links,ok,hard_dead,uncertain,ok_pct,hard_dead_pct,uncertain_pct,hard_dead_ci_lo,hard_dead_ci_hi,broken_floor_pct,broken_ceiling_pct
source,,,,,,,,,,,
not harvested,2239,660,96,1483,29.5,4.3,66.2,3.5,5.2,4.3,70.5
harvested,250,0,0,250,0.0,0.0,100.0,0.0,1.5,0.0,100.0


## Verdict against the corridor

For each publisher I put the honest band, from the floor (confirmed dead only) to the ceiling (if every
uncertain link were dead), next to the national ~7% after and ~30% before. A publisher sits
*materially outside* only if its whole band clears a reference line, or if the 95% interval on its
confirmed-dead share does.

In [16]:
v = final[["links", "broken_floor_pct", "broken_ceiling_pct", "hard_dead_ci_lo", "hard_dead_ci_hi", "uncertain_pct"]].copy()
def verdict(r):
    if r.links < 30:
        return "too few links to judge"
    if r.hard_dead_ci_lo > 30:
        return "confirmed-dead alone is ABOVE the national 'before' rate"
    if r.hard_dead_ci_lo > 7:
        return "confirmed-dead alone is ABOVE the national 'after' rate"
    if r.broken_ceiling_pct < 7:
        return "below 7% even if every uncertain link is dead"
    if r.uncertain_pct > 25:
        return "indeterminate: uncertain share too big to place it"
    return "inside the corridor"
v["verdict"] = v.apply(verdict, axis=1)
v.to_csv(config.GOLD / "verdict.csv"); v.round(1)

final_class,links,broken_floor_pct,broken_ceiling_pct,hard_dead_ci_lo,hard_dead_ci_hi,uncertain_pct,verdict
publisher,,,,,,,
"NHS Digital (merged into NHS England, 2023)",2489,3.9,73.5,3.2,4.7,69.6,indeterminate: uncertain share too big to plac...
Department of Health and Social Care,1160,0.9,66.6,0.5,1.6,65.7,indeterminate: uncertain share too big to plac...
NHS Business Services Authority,198,0.0,0.0,0.0,1.9,0.0,below 7% even if every uncertain link is dead
NHS England,439,0.0,4.6,0.0,0.9,4.6,below 7% even if every uncertain link is dead
Public Health England (closed 2021; to UKHSA/OHID),17,0.0,0.0,0.0,18.4,0.0,too few links to judge
UK Health Security Agency,2,0.0,0.0,0.0,65.8,0.0,too few links to judge
